In [ ]:
## 2026.08.20 CODEX GIST counterpart of CODEX_hcc/HCC_histology_derived_niche_index.ipynb
## TLS / SRI / TNI from StarDist Level-2 softmax on s1167 matched nuclei
## Helpers: code/CODEX_gist/gist_histology_derived_niche_index.py


## 1. Load predicted cell-type probabilities (soft abundance)

This notebook is the **CODEX GIST (s1167)** analog of `CODEX_hcc/HCC_histology_derived_niche_index.ipynb`.

It uses pooled StarDist matched-AUROC predictions under `s1167/result_all_spatial/stardist/{ACQUISITION_ID}/`.

| Source | Predictions | Coordinates |
|--------|-------------|-------------|
| **Matched AUROC** | `stardist/{acq}/validation_external_stardist_matched_AUROC.csv` | `s1167/{acq}/{acq}_matched_features_stardist.h5ad` (`obsm['spatial_HE']`) |

**Demo core:** `Charvill-94_c013_v001_r001_reg002`. Cohort = annotated GIST cores (550).

There is **no Response** field. Clinical groups are **coverslip** and **SAMPLE_LABEL**.

GIST Level-2 adds Endothelial cells, Stromal cells, T cells, Monocytes, DCs on top of the PDAC set.

Key helpers: `gist_histology_derived_niche_index.py` (compartments) + `Xenium_lung/histology_derived_niche_index.py` (spatial/plot).


In [ ]:
import sys
import importlib
from pathlib import Path

import matplotlib
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

path = "/home/lingyu/ssd2/Python/"
_repo = Path(f"{path}Hist2Pheno/code")
for _p in (_repo / "Hist2Pheno_pkg", _repo / "Xenium_lung", _repo / "CODEX_pdac", _repo / "CODEX_gist"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import gist_histology_derived_niche_index as gist_ni
importlib.reload(gist_ni)

DATA_ROOT = gist_ni.DEFAULT_DATA_ROOT
CASES_ROOT = gist_ni.DEFAULT_CASES_ROOT
STARDIST_ROOT = gist_ni.DEFAULT_STARDIST_ROOT
OUT_DIR = DATA_ROOT / "result_all_spatial" / "niche_index_gist"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("STARDIST :", STARDIST_ROOT)
print("OUT_DIR  :", OUT_DIR)
print("pan_organ:", gist_ni.PAN_ORGAN)


In [ ]:
SAMPLE = gist_ni.DEFAULT_DEMO_SAMPLE
RADIUS_UM = gist_ni.DEFAULT_SPATIAL_RADIUS_UM
UM_PER_PX = gist_ni.DEFAULT_UM_PER_HE_PIXEL

clinical = gist_ni.load_s1167_clinical_info()
annotation = gist_ni.load_s1167_annotation()
ALL_SAMPLES = gist_ni.discover_s1167_spatial_samples(clinical, require_h5ad=True)
paths = gist_ni.s1167_spatial_paths(SAMPLE)

print(f"Demo core: {SAMPLE}")
print(f"Annotated cores with AUROC+h5ad: {len(ALL_SAMPLES)}")
print("AUROC:", paths.auroc_csv)
print("h5ad :", paths.stardist_h5ad)
print(f"Spatial radius: {RADIUS_UM} μm | μm/px={UM_PER_PX}")
display(gist_ni.index_formula_table())
display(annotation)
display(clinical.head())


In [ ]:
df, class_names, probs = gist_ni.load_stardist_auroc_csv(
    paths.auroc_csv, paths.class_names_csv,
)
missing = gist_ni.missing_index_cell_types(class_names)
print(f"{SAMPLE}: {len(df):,} cells, {len(class_names)} classes")
print("classes:", class_names)
if missing:
    raise KeyError(f"Missing index compartments in class_names: {missing}")
df.head(3)


## 2. Compute histology-derived indices (abundance)

GIST indices use **Level-2 softmax** (soft abundance per nucleus).

| Index | Formula | HCC analog |
|-------|---------|------------|
| **TLS** | B + T cells (+ Cytotoxic/Helper/Tregs if present) + DC/DCs | TLS |
| **SRI_ratio** | (Fibroblasts + Stromal cells) / Epithelial cells | SRI |
| **TNI_ratio** | (Fib/Stromal + Macrophages/Monocytes + Endothelial) / epithelium | TNI |

**Read order:** SRI → TNI → TLS. High SRI alone = epithelial mass replaced by stroma. High SRI **and** TNI = active CAF / macrophage / endothelial niche.


In [ ]:
df_idx = gist_ni.add_abundance_indices(df, probs, class_names)
sample_summary = gist_ni.summarize_indices_by_sample(df_idx, SAMPLE)
display(sample_summary)
df_idx.filter(regex="^(cell_id|predict_stardist|idx_)").head()


## 3. Spatial TLS (B / T / DC co-localization)

**TLS_spatial** = `(B_local × T_local × DC_local)^(1/3)` within radius **R μm**.

Cyan points = `tls_candidate` (top 5% `idx_TLS_spatial` on this core).


In [ ]:
df_spatial, class_names_sp, probs_sp, coords, paths_sp, um_px = gist_ni.load_sample_with_spatial_indices(
    SAMPLE, radius_um=RADIUS_UM, um_per_pixel=UM_PER_PX,
)
print(f"{SAMPLE} | μm/px={um_px:.4f} | R={RADIUS_UM} μm | n={len(df_spatial):,}")
print("TLS hotspot:", gist_ni.summarize_tls_hotspots(df_spatial))
cols = [
    "cell_id", "coord_x", "coord_y", "idx_TLS", "idx_TLS_spatial",
    "idx_SRI_spatial", "idx_TNI_spatial", "tls_candidate", "sri_candidate", "tni_candidate",
]
display(
    df_spatial.loc[df_spatial["tls_candidate"], cols]
    .sort_values("idx_TLS_spatial", ascending=False)
    .head(5)
)


In [ ]:
gist_ni.plot_tls_abundance_vs_spatial(
    df_spatial, f"{SAMPLE} (CODEX GIST)",
    radius_um=RADIUS_UM, figsize=(18.0 * 1.2, 5.0 * 1.2), dpi=150,
)


## 4. Spatial SRI / TNI (stromal remodeling → active TME niche)

1. **SRI** — local fibroblasts vs epithelium.
2. **TNI** — co-localized fibroblasts × macrophages × endothelium / epithelium.
3. **TLS** — adaptive immune aggregate.

High SRI **and** high TNI → active immunosuppressive / angiogenic microenvironment.


In [ ]:
display(
    df_spatial[
        ["cell_id", "coord_x", "coord_y", "idx_SRI_ratio", "idx_SRI_spatial",
         "idx_TNI_ratio", "idx_TNI_spatial", "sri_candidate", "tni_candidate"]
    ]
    .sort_values("idx_TNI_spatial", ascending=False)
    .head(5)
)
gist_ni.plot_sri_tni_abundance_vs_spatial(
    df_spatial, f"{SAMPLE} (CODEX GIST)",
    radius_um=RADIUS_UM, figsize=(18.0 * 1.2, 10.0 * 1.2), dpi=150,
)
active = df_spatial["sri_candidate"] & df_spatial["tni_candidate"]
print(
    f"Active TME-niche cells (SRI ∩ TNI hotspot): {int(active.sum())} "
    f"({100 * active.mean():.2f}% of tissue)"
)


## 5. Hierarchical SRI → TNI (within-core quadrants)

Biological hierarchy: **Normal (Q1) → SRI↑ remodeling (Q2) → SRI↑+TNI↑ active niche (Q4)**.

Cross-core clinical comparisons use **Global Q4** and **active niche burden** in §6.


In [ ]:
sri_tni_result = gist_ni.analyze_sri_tni_relationship(
    df_spatial, SAMPLE, plot=True, plot_layout="combined",
    figsize=(14, 6), figsize_scatter=(5.5, 5.5), figsize_spatial=(8.0, 8.0), dpi=150,
)
display(sri_tni_result["quadrant_table"])
print(
    f"Spearman rho={sri_tni_result['Spearman_rho']:.3f}, p={sri_tni_result['Spearman_p']:.2g} | "
    f"Q4 (active TME niche)={sri_tni_result['Q4_percent']:.2f}%"
)


## 6. Cohort: TME niche burden vs coverslip / SAMPLE_LABEL

| Metric | Definition |
|--------|------------|
| **Global Q4 %** | % cells with SRI and TNI ≥ cohort-pooled 95th percentiles |
| **Active niche burden** | `mean(√(SRI_i × TNI_i))` over matched nuclei |

Clinical groups: **coverslip**, **SAMPLE_LABEL** (`pan_organ="codex_gist"`).

Set `QUICK_VALIDATE = True` to load a few cores first. Full 550-core run is slow.


In [ ]:
import importlib
importlib.reload(gist_ni)

QUICK_VALIDATE = True
COHORT_SAMPLES = ALL_SAMPLES[:6] if QUICK_VALIDATE else ALL_SAMPLES
print(f"Loading {len(COHORT_SAMPLES)} cores (QUICK_VALIDATE={QUICK_VALIDATE})")

sample_dict = gist_ni.build_sample_dict(
    COHORT_SAMPLES, radius_um=RADIUS_UM, um_per_pixel=UM_PER_PX,
)
sri_tni_batch, cohort_meta = gist_ni.batch_analyze_sri_tni_cohort(sample_dict)
print(
    f"Cohort global cutoffs (p{cohort_meta['percentile']:.0f}, "
    f"n={cohort_meta['n_cells_pooled']:,} cells): "
    f"SRI={cohort_meta['sri_threshold_global']:.4f}, "
    f"TNI={cohort_meta['tni_threshold_global']:.4f}"
)
display(
    sri_tni_batch.sort_values("active_niche_burden", ascending=False)[
        ["Sample", "n_cells", "Q4_global_percent", "active_niche_burden",
         "SRI_median", "TNI_median", "Spearman_rho"]
    ].head()
)


In [ ]:
sri_tni_clinical = gist_ni.merge_batch_with_clinical(sri_tni_batch, clinical)
q4_stats = gist_ni.test_clinical_groups(
    sri_tni_clinical, metric_col="Q4_global_percent", alternative="two-sided",
)
burden_stats = gist_ni.test_clinical_groups(
    sri_tni_clinical, metric_col="active_niche_burden", alternative="two-sided",
)
print("=== Global Q4 % ===")
display(q4_stats)
print("=== Active TME niche burden ===")
display(burden_stats)

gist_ni.plot_clinical_comparison(
    sri_tni_clinical, burden_stats,
    q4_col="active_niche_burden",
    figsize=(10, 4), legend_ncol=2,
    ylabel="Active TME niche burden",
    suptitle="CODEX GIST active TME niche burden by coverslip / SAMPLE_LABEL",
)


## 7. SAMPLE_LABEL-level source decomposition

GIST has no immunotherapy Response. This section:

1. Region-level **SRI_mean**, **TNI_mean**, **Epi_local_mean**, **TNI_numerator_mean**.
2. Tests those metrics vs **coverslip** / **SAMPLE_LABEL**.
3. Averages within `SAMPLE_LABEL` (TMA donor analog) and re-tests.

Requires `sample_dict` from §6.


In [ ]:
source = gist_ni.summarize_source_metrics(sample_dict)
region_source = sri_tni_clinical.merge(
    source.drop(columns=["n_cells"], errors="ignore"), on="Sample", how="left",
)
print("Cores:", len(region_source), "| SAMPLE_LABEL:", region_source["SAMPLE_LABEL"].nunique())
display(
    region_source[
        ["Sample", "SAMPLE_LABEL", "coverslip", "SRI_mean", "TNI_mean",
         "Epi_local_mean", "TNI_numerator_mean", "active_niche_burden"]
    ].head()
)
region_source_stats = gist_ni.test_source_metrics(region_source)
print("=== Core-level clinical tests ===")
display(region_source_stats)

patient_source = gist_ni.aggregate_metrics_by_patient(region_source)
patient_source_stats = gist_ni.test_source_metrics(patient_source)
print("=== SAMPLE_LABEL-level (mean of cores) ===")
display(patient_source)
display(patient_source_stats)


In [ ]:
gist_ni.plot_source_metric_grid(
    region_source, region_source_stats,
    sample_col="Sample", group_col="SAMPLE_LABEL",
    suptitle="Core-level (points colored by ACQUISITION_ID)",
)
gist_ni.plot_source_metric_grid(
    patient_source, patient_source_stats,
    sample_col="SAMPLE_LABEL", group_col="coverslip",
    suptitle="SAMPLE_LABEL-level (one point per TMA donor)",
)
